# Using External Tools with Model Context Protocol (MCP)

## Create the MCP Client

In [1]:
from langchain_mcp_adapters.client import MultiServerMCPClient

deep_wiki_client = MultiServerMCPClient(
    {
        "deepwiki": {
            "transport": "http",
            "url": "https://mcp.deepwiki.com/mcp",
        }
    }
)

取得工具清單
會轉成 LangChain 的 StructuredTool 物件，並且可以直接使用 LangChain 的 Agent 來使用這些工具。

In [2]:
tools = await deep_wiki_client.get_tools()

In [3]:
from pprint import pprint

pprint(tools)

[StructuredTool(name='ask_question', description='Ask any question about a GitHub repository and get an AI-powered, context-grounded response.', args_schema={'properties': {'repoName': {'anyOf': [{'type': 'string'}, {'items': {'type': 'string'}, 'type': 'array'}], 'description': 'GitHub repository or list of repositories (max 10) in owner/repo format.'}, 'question': {'description': 'The question to ask about the repository.', 'type': 'string'}}, 'required': ['repoName', 'question'], 'type': 'object'}, metadata={'_meta': {'_fastmcp': {'tags': []}}}, handle_tool_error=<function _handle_mcp_tool_error at 0x10aa17920>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x10aae1940>),
 StructuredTool(name='read_wiki_contents', description='View documentation about a GitHub repository.', args_schema={'properties': {'repoName': {'description': 'GitHub repository in owner/repo format (e.g. "facebook/react").', 'type': 'string'}}

In [4]:
pprint(dict(tools[0]))

{'args_schema': {'properties': {'question': {'description': 'The question to '
                                                            'ask about the '
                                                            'repository.',
                                             'type': 'string'},
                                'repoName': {'anyOf': [{'type': 'string'},
                                                       {'items': {'type': 'string'},
                                                        'type': 'array'}],
                                             'description': 'GitHub repository '
                                                            'or list of '
                                                            'repositories (max '
                                                            '10) in owner/repo '
                                                            'format.'}},
                 'required': ['repoName', 'question'],
                 'type': 

In [5]:
# Print all tool information
for tool in tools:
    pprint(f"Tool name: {tool.name}")
    pprint(f"Tool description: {tool.description}")
    pprint(f"Tool input schema:")
    pprint(tool.args_schema)
    pprint("-" * 40)

'Tool name: ask_question'
('Tool description: Ask any question about a GitHub repository and get an '
 'AI-powered, context-grounded response.')
'Tool input schema:'
{'properties': {'question': {'description': 'The question to ask about the '
                                            'repository.',
                             'type': 'string'},
                'repoName': {'anyOf': [{'type': 'string'},
                                       {'items': {'type': 'string'},
                                        'type': 'array'}],
                             'description': 'GitHub repository or list of '
                                            'repositories (max 10) in '
                                            'owner/repo format.'}},
 'required': ['repoName', 'question'],
 'type': 'object'}
'----------------------------------------'
'Tool name: read_wiki_contents'
'Tool description: View documentation about a GitHub repository.'
'Tool input schema:'
{'properties': {'repoName':

## Create the agent with the tools

In [10]:
system_prompt = """
You are an assistant for studying the source code of LangChain.
You use the DeepWiki MCP tools to answer questions about the LangChain repository.

Target repository in the format of owner/repository is:
- langchain-ai/langchain

Mandatory behavior:
- For questions about LangChain implementation, architecture, modules,
  classes, functions, or execution flow, call a DeepWiki MCP tool before
  answering.
- Do not use the pre-trained knowledge of the model to answer questions about LangChain repository-specific questions.
- Use ask_question for a focused implementation question.
- Use read_wiki_structure to discover available documentation sections.
- Use read_wiki_contents to inspect a relevant section.
- If DeepWiki does not provide enough evidence, state that limitation.
- Answer in Traditional Chinese.
"""

In [11]:
from langchain.agents import create_agent

lanchain_doc_agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt=system_prompt)

Ask questions

In [12]:
question_1 = "請讀取 langchain repo 的 Wiki structure，列出與 Agent 相關的主要章節。"
state = {"messages": [{"role": "user", "content": question_1}]}

In [13]:
response_1 = await lanchain_doc_agent.ainvoke(state)

In [14]:
pprint(response_1)

{'messages': [HumanMessage(content='請讀取 langchain repo 的 Wiki structure，列出與 Agent 相關的主要章節。', additional_kwargs={}, response_metadata={}, id='3f55c3d9-ce41-469a-9cb2-17235fe0be40'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 350, 'prompt_tokens': 441, 'total_tokens': 791, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E68O24QA9nxfxg7x5rIxUtFb7vJGB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fa218-0b50-7210-8167-41f46187712a-0', tool_calls=[{'name': 'read_wiki_structure', 'args': {'repoName': 'langchain-ai/langchain'}, 'id': 'call_3MNPIHAnYgW7UC20QJKCSDXt', 't

In [15]:
# Print the response
pprint(response_1["messages"][-1].content)

('以下是 Wiki 結構中與 Agent 相關的主要章節（位於 4. Agent System）：\n'
 '\n'
 '- 4. Agent System\n'
 '  - 4.1 Agent Creation and Middleware Architecture\n'
 '  - 4.2 Middleware Implementations\n'
 '  - 4.3 Structured Output and Response Formats\n'
 '  - 4.4 Configuration and Runtime Control\n'
 '\n'
 '如需，我可以進一步檢視各小節的內容或整理成摘要。')


In [17]:
# 查看 Tool Calling 過程
from langchain.messages import AIMessage, ToolMessage
for index, message in enumerate(response_1["messages"]):
    print(f"\n[{index}] {type(message).__name__}")
    if isinstance(message, AIMessage):
        print("Content:", message.content)
        print("Tool calls:", message.tool_calls)
    elif isinstance(message, ToolMessage):
        print("Tool name:", message.name)
        print("Tool call ID:", message.tool_call_id)
        print("Content:", message.content)


[0] HumanMessage

[1] AIMessage
Content: 
Tool calls: [{'name': 'read_wiki_structure', 'args': {'repoName': 'langchain-ai/langchain'}, 'id': 'call_3MNPIHAnYgW7UC20QJKCSDXt', 'type': 'tool_call'}]

[2] ToolMessage
Tool name: read_wiki_structure
Tool call ID: call_3MNPIHAnYgW7UC20QJKCSDXt
Content: [{'type': 'text', 'text': 'Available pages for langchain-ai/langchain:\n\n- 1 LangChain Overview\n  - 1.1 Package Ecosystem\n- 2 Core Architecture\n  - 2.1 Runnable Interface and LCEL\n  - 2.2 Language Models and Chat Models\n  - 2.3 Tools and Function Calling\n  - 2.4 Messages and Content Handling\n  - 2.5 Prompt Templates\n  - 2.6 Text Splitters\n  - 2.7 Callbacks and Event System\n  - 2.8 Output Parsers and Serialization\n  - 2.9 Storage, Indexing, and Vector Stores\n- 3 Provider Integrations\n  - 3.1 OpenAI and Anthropic Integrations\n  - 3.2 Groq, Fireworks, and OpenRouter Integrations\n  - 3.3 Ollama Integration\n  - 3.4 Embeddings and Vector Stores\n  - 3.5 Integration Patterns and Best